# RAG Temperature Experiment - Randomized Complete Block Design (RCBD)

**Purpose:** Test how LLM temperature affects Rule Grounding Score (RGS) in a retrieval-augmented AI agent

**Design:** Randomized Complete Block Design (RCBD)
- **Treatment Factor:** Temperature (3 levels: 0.3, 0.5, 0.8)
- **Blocking Factor:** Question Type (3 blocks: S, M, F)
- **Experimental Units:** 90 total runs
  - 30 questions (10 per block)
  - Each question tested at all 3 temperatures
  - Complete randomization of run order
- **Response Variable:** RGS = C1 × (C2 + C3 + C4 + C5) / 4

**RCBD Advantages:**
- Controls for question difficulty variance via blocking
- Each question serves as its own control across temperatures
- Complete randomization prevents order/time confounding

**Workflow:**
1. Load 30 questions (10 per question type)
2. Create design matrix: 30 questions × 3 temperatures = 90 runs
3. Completely randomize run order
4. Execute experiment → CSV with responses
5. Score C1-C5 manually (or with LLM-as-judge)
6. Calculate RGS
7. Import to JMP for ANOVA (with blocking)

## 1. Setup and Imports

In [21]:
# Core imports
import os
import sys
import json
import random
import time
import csv
import re
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# API clients
import voyageai
from anthropic import Anthropic
from anthropic.types import Message

# Exception handling - with fallback for older SDK versions
try:
    from anthropic import APITimeoutError, APIConnectionError, RateLimitError
except ImportError:
    APITimeoutError = TimeoutError
    APIConnectionError = ConnectionError
    RateLimitError = Exception

# Custom retriever implementation
sys.path.append("..")
from retriever_implementation import VectorIndex, BM25Index, Retriever

print("✓ Imports successful")

✓ Imports successful


In [22]:
# Load environment and initialize clients
load_dotenv(dotenv_path='../.env')

# VoyageAI for embeddings
v_api_key = os.getenv('VOYAGE_API_KEY')
embedding_client = voyageai.Client(api_key=v_api_key)

# Anthropic for LLM
client = Anthropic()
model = os.getenv('MODEL_NAME')
max_tokens = int(os.getenv('MAX_TOKENS'))

print("✓ API clients initialized")
print(f"  Model: {model}")

✓ API clients initialized
  Model: claude-haiku-4-5-20251001


In [23]:
# Temperature levels for experiment
temp = {
    "low": 0.3,
    "medium": 0.5,
    "high": 0.8
}

print(f"✓ Temperature levels defined: {temp}")

✓ Temperature levels defined: {'low': 0.3, 'medium': 0.5, 'high': 0.8}


## 2. Helper Functions
These are copied from Anthropic's Learning 

In [24]:
# Message handling helpers
def add_user_message(messages, message):
    """Add a user message to the conversation"""
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    """Add an assistant message to the conversation"""
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)

def chat(messages, system=None,  temperature=0.5, stop_sequences=[], tools=None, timeout=60):
    """Call Claude API with timeout support"""
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
        "timeout": timeout
    }
    
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    
    return client.messages.create(**params)

def text_from_message(message):
    """Extract text content from Claude response"""
    return "\n".join([block.text for block in message.content if block.type == "text"])

print("✓ Helper functions defined")

✓ Helper functions defined


## 3. RAG Setup

In [25]:
# Document chunking
def chunk_by_section(document_text):
    """Split document by ## headers"""
    pattern = r"\n## "
    return re.split(pattern, document_text)

print("✓ Chunking function defined")

✓ Chunking function defined


In [26]:
# Embedding generation
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    """Generate embeddings using VoyageAI"""
    is_list = isinstance(chunks, list)
    input_data = chunks if is_list else [chunks]
    result = embedding_client.embed(input_data, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

print("✓ Embedding function defined")

✓ Embedding function defined


In [27]:
# Reranker
def reranker_fn(docs, query_text, k):
    """Use Claude to rerank retrieved documents"""
    joined_docs = "\n".join([
        f"""
        <document>
        <document_id>{doc["id"]}</document_id>
        <document_content>{doc["content"]}</document_content>
        </document>
        """
        for doc in docs
    ])
    
    prompt = f"""
    You are about to be given a set of documents, along with an id of each.
    Your task is to select the {k} most relevant documents to answer the user's question.

    Here is the user's question:
    <question>
    {query_text}
    </question>
    
    Here are the documents to select from:
    <documents>
    {joined_docs}
    </documents>

    Respond in the following format:
    ```json
    {{
        "document_ids": str[]
    }}
    ```
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    
    result = chat(messages, stop_sequences=["```"])
    return json.loads(text_from_message(result))["document_ids"]

print("✓ Reranker function defined")

✓ Reranker function defined


In [28]:
# Contextual chunk enhancement
def add_context(text_chunk, source_text):
    """Add context to a chunk for better retrieval"""
    prompt = f"""
    Write a short and succinct snippet of text to situate this chunk within the 
    overall source document for the purposes of improving search retrieval of the chunk. 

    Here is the original source document:
    <document> 
    {source_text}
    </document> 

    Here is the chunk we want to situate within the whole document:
    <chunk> 
    {text_chunk}
    </chunk>
    
    Answer only with the succinct context and nothing else. 
    """
    
    messages = []
    add_user_message(messages, prompt)
    result = chat(messages)
    
    return text_from_message(result) + "\n" + text_chunk

print("✓ Context enhancement function defined")

✓ Context enhancement function defined


In [29]:
# Load and process the Clue rules document
with open("../References/ClueRules.md", "r") as f:
    clue_instructions = f.read()

chunks = chunk_by_section(clue_instructions)
print(f"✓ Loaded Clue rules and created {len(chunks)} chunks")

✓ Loaded Clue rules and created 8 chunks


In [30]:
# Print each chunk with its number
with open("../Output/ChunkedInstructins.txt", "w", encoding='utf-8') as f:
    for i, chunk in enumerate(chunks, 1):
        f.write(f"\n{'='*70}\n")      # ← Added \n at end
        f.write(f"CHUNK {i}\n")       # ← Added \n at end
        f.write(f"{'='*70}\n")        # ← Added \n at end
        f.write(chunk)
        f.write('\n----------\n')     # ← Added \n before and after

In [31]:
# Create retriever
vector_index = VectorIndex(embedding_fn=generate_embedding)
bm25_index = BM25Index()
retriever = Retriever(bm25_index, vector_index, reranker_fn=reranker_fn)

print("✓ Retriever created")

✓ Retriever created


In [ ]:
# Add contextualized chunks to retriever
num_start_chunks = 2
num_prev_chunks = 2
contextualized_chunks = []

print("Adding contextualized chunks to retriever...")
for i, chunk in enumerate(chunks):
    context_parts = []
    context_parts.extend(chunks[: min(num_start_chunks, len(chunks))])
    start_idx = max(0, i - num_prev_chunks)
    context_parts.extend(chunks[start_idx:i])
    context = "\n".join(context_parts)
    
    contextualized_chunks.append(add_context(chunk, context))
    print(f"  Processed chunk {i+1}/{len(chunks)}")

retriever.add_documents([
    {
        "content": chunk,
        "chunk_number": i+1  # ← Add the actual chunk number!
    } 
    for i, chunk in enumerate(contextualized_chunks)
])
print(f"✓ Added {len(contextualized_chunks)} contextualized chunks to retriever")

Adding contextualized chunks to retriever...
  Processed chunk 1/8
  Processed chunk 2/8
  Processed chunk 3/8
  Processed chunk 4/8
  Processed chunk 5/8
  Processed chunk 6/8
  Processed chunk 7/8
  Processed chunk 8/8
✓ Added 8 contextualized chunks to retriever


## 4. RAG Tool Definition

In [ ]:
def search_clue_instructions(query, k=3): #retieval coverage k:chunks or Recall@k
    """
    Search the Clue game instructions with citation metadata.
    
    Args:
        query (str): Search query
        k (int): Number of chunks to return
    
    Returns:
        str: JSON string with search results
    """
    results = retriever.search(query, k=k)
    
    formatted_results = []
    for i, (doc, score) in enumerate(results):
        chunk_content = doc["content"]
        preview = chunk_content[:200].strip()
        if len(chunk_content) > 200:
            preview += "..."
        
        formatted_results.append({
            "chunk_id": f"CHUNK_{doc['chunk_number']}",  # ← FIXED!
            "content": chunk_content,
            "preview": preview,
            "relevance_score": float(score)
        })
    
    return json.dumps(formatted_results, indent=2)
print("✓ Search function defined")

In [34]:
# Tool schema for Claude
tools = [
    {
        "name": "search_clue_instructions",
        "description": "Search the official Clue game instructions to find information about rules, gameplay, setup, winning conditions, and player counts. Use this tool whenever you need specific information from the Clue game manual.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query - what information to look for in the Clue instructions (e.g., 'player count', 'how to win', 'setup rules')"
                }
            },
            "required": ["query"]
        }
    }
]

print("✓ Tool schema defined")

✓ Tool schema defined


## 5. Agent with Retry Logic

In [35]:
def answer_clue_question(user_question, temperature_value=0.5, max_iterations=5, max_retries=3, retry_delay=5):
    """
    Answer a question about Clue using RAG via tool use.
    Includes timeout and retry logic.
    
    Args:
        user_question (str): The user's question
        temperature_value (float): Temperature for this run
        max_iterations (int): Max tool-use cycles
        max_retries (int): Max retry attempts on errors
        retry_delay (int): Seconds between retries
    
    Returns:
        dict: {"answer": str, "iterations": int, "retries": int, "error": str or None, "tool_calls": list}
    """
    
    system_prompt = """
<role>
You are a fun and enthusiastic game instructor who teaches people how to play Clue.
</role>

<capabilities>
<can_answer>
- Rules and game mechanics
- Setup instructions
- Turn structure
- Win conditions
- Card and token mechanics
- Suggestions and accusations
</can_answer>

<cannot_answer>
- Strategy advice
- Probability calculations
- Tactical recommendations
- Player psychology
</cannot_answer>
</capabilities>

<tools>
<tool_name>search_clue_instructions</tool_name>
<when_to_use>Whenever you need specific information from the official Clue game rules</when_to_use>
</tools>

<response_format>
<for_rules_questions>
1. Use search_clue_instructions tool
2. Provide clear, friendly explanation
3. Cite sources: "According to CHUNK_X..."
4. Include brief direct quote
</for_rules_questions>

<for_strategy_questions>
Politely decline and redirect
</for_strategy_questions>

<for_off_topic_questions>
"I only answer questions about the game Clue."
</for_off_topic_questions>
</response_format>
"""
    
    retry_count = 0
    
    while retry_count <= max_retries:
        try:
            messages = []
            add_user_message(messages, user_question)
            iteration_count = 0
            tool_calls = []
            
            for iteration in range(max_iterations):
                iteration_count = iteration + 1
                print(f"--- Iteration {iteration + 1} ---")
                
                response = chat(
                    messages=messages,
                    system=system_prompt,
                    temperature=temperature_value,
                    tools=tools,
                    timeout=60
                )
                
                print(f"Stop reason: {response.stop_reason}")
                
                if response.stop_reason == "end_turn":
                    return {
                        "answer": text_from_message(response),
                        "iterations": iteration_count,
                        "retries": retry_count,
                        "error": None,
                        "tool_calls": tool_calls
                    }
                
                if response.stop_reason == "tool_use":
                    add_assistant_message(messages, response)
                    
                    for content_block in response.content:
                        if content_block.type == "tool_use":
                            tool_name = content_block.name
                            tool_input = content_block.input
                            tool_use_id = content_block.id
                            
                            tool_calls.append({
                                "tool": tool_name,
                                "query": tool_input.get("query", "")
                            })
                            
                            print(f"Claude wants to use: {tool_name}")
                            print(f"With query: {tool_input['query']}")
                            
                            try:
                                if tool_name == "search_clue_instructions":
                                    tool_result = search_clue_instructions(tool_input["query"])
                                else:
                                    tool_result = json.dumps({"error": f"Unknown tool: {tool_name}"})
                                
                                print(f"Tool result preview: {tool_result[:150]}...")
                            
                            except Exception as tool_error:
                                print(f"⚠️ Tool execution error: {str(tool_error)[:100]}")
                                tool_result = json.dumps({
                                    "error": f"Tool execution failed: {str(tool_error)}",
                                    "tool": tool_name
                                })
                            
                            messages.append({
                                "role": "user",
                                "content": [{
                                    "type": "tool_result",
                                    "tool_use_id": tool_use_id,
                                    "content": tool_result
                                }]
                            })
                    
                    continue
                
                return {
                    "answer": text_from_message(response),
                    "iterations": iteration_count,
                    "retries": retry_count,
                    "error": None,
                    "tool_calls": tool_calls
                }
            
            return {
                "answer": "I apologize, but I'm having trouble processing this question.",
                "iterations": iteration_count,
                "retries": retry_count,
                "error": "max_iterations_reached",
                "tool_calls": tool_calls
            }
        
        except (APITimeoutError, APIConnectionError, TimeoutError, ConnectionError) as e:
            error_name = type(e).__name__.lower()
            error_type = "timeout" if "timeout" in error_name else "connection"
            
            if iteration_count > 0:
                print(f"❌ {error_type.capitalize()} error mid-conversation - cannot retry")
                return {
                    "answer": "",
                    "iterations": iteration_count,
                    "retries": retry_count,
                    "error": f"{error_type}_mid_conversation: {str(e)}",
                    "tool_calls": tool_calls if 'tool_calls' in locals() else []
                }
            
            retry_count += 1
            
            if retry_count <= max_retries:
                print(f"⚠️ {error_type.capitalize()} error (attempt {retry_count}/{max_retries+1})")
                print(f"   Waiting {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                print(f"❌ Max retries exceeded")
                return {
                    "answer": "",
                    "iterations": 0,
                    "retries": retry_count - 1,
                    "error": f"{error_type}_error: {str(e)}",
                    "tool_calls": []
                }
        
        except RateLimitError as e:
            print(f"❌ Rate limit error: {str(e)}")
            return {
                "answer": "",
                "iterations": 0,
                "retries": retry_count,
                "error": f"rate_limit_error: {str(e)}",
                "tool_calls": []
            }
        
        except Exception as e:
            error_name = type(e).__name__.lower()
            if "ratelimit" in error_name or "rate_limit" in error_name:
                print(f"❌ Rate limit error: {str(e)}")
                return {
                    "answer": "",
                    "iterations": 0,
                    "retries": retry_count,
                    "error": f"rate_limit_error: {str(e)}",
                    "tool_calls": []
                }
            
            print(f"❌ Unexpected error: {str(e)}")
            return {
                "answer": "",
                "iterations": 0,
                "retries": retry_count,
                "error": f"unexpected_error: {str(e)}",
                "tool_calls": []
            }

print("✓ Agent function defined with timeout/retry logic")

✓ Agent function defined with timeout/retry logic


## 6. Load Experimental Questions

In [36]:
def load_questions_from_csv(csv_path):
    """
    Load questions with their types from CSV.
    
    Returns:
        list: [{"number": 1, "question": "...", "type": "S"}, ...]
    """
    df = pd.read_csv(csv_path)
    
    questions = []
    for _, row in df.iterrows():
        questions.append({
            "number": int(row['Number']),
            "question": row['Question'],
            "type": row['Type']
        })
    
    print(f"✅ Loaded {len(questions)} questions from {csv_path}")
    
    type_counts = df['Type'].value_counts().to_dict()
    print(f"   Question types: S={type_counts.get('S', 0)}, "
          f"M={type_counts.get('M', 0)}, F={type_counts.get('F', 0)}")
    
    return questions

# Load questions
questions = load_questions_from_csv("../References/ClueQuestions.csv")

✅ Loaded 31 questions from ../References/ClueQuestions.csv
   Question types: S=10, M=10, F=11


## 7. RCBD Experiment Function

**Key RCBD Features:**
1. **Design matrix creation**: All 90 (question, temperature) combinations created upfront
2. **Complete randomization**: Entire design matrix shuffled before execution
3. **Blocking tracked**: `block` column contains question type (S, M, F)
4. **No systematic order effects**: Temperature assignment is random across all runs

In [ ]:
def run_RCBD_temperature_experiment(
    questions_data,
    temperature_dict,
    csv_filename="clue_temperature_RCBD.csv",
    delay_seconds=3
):
    """
    Run RCBD temperature experiment with complete randomization.
    
    Args:
        questions_data (list): Questions from load_questions_from_csv()
        temperature_dict (dict): Temperature levels to test
        csv_filename (str): Output CSV file
        delay_seconds (int): Delay between API calls
    
    Returns:
        dict: Experiment statistics
    """
    
    # Step 1: Create complete design matrix
    print("\n" + "="*70)
    print("CREATING RCBD DESIGN MATRIX")
    print("="*70)
    
    design_matrix = []
    for q_data in questions_data:
        for temp_label, temp_value in temperature_dict.items():
            design_matrix.append({
                "question_number": q_data["number"],
                "question_text": q_data["question"],
                "question_type": q_data["type"],
                "block": q_data["type"],  # Blocking variable
                "temperature_level": temp_label,
                "temperature_value": temp_value
            })
    
    print(f"Design matrix created: {len(design_matrix)} runs")
    print(f"  {len(questions_data)} questions × {len(temperature_dict)} temperatures")
    
    # Verify blocking structure
    block_counts = {}
    for entry in design_matrix:
        block = entry["block"]
        block_counts[block] = block_counts.get(block, 0) + 1
    
    print(f"\nBlocking structure:")
    for block, count in sorted(block_counts.items()):
        questions_in_block = count // len(temperature_dict)
        print(f"  Block {block}: {questions_in_block} questions × {len(temperature_dict)} temps = {count} runs")
    
    # Step 2: Complete randomization
    print(f"\n🔀 Randomizing all {len(design_matrix)} runs...")
    random.shuffle(design_matrix)
    print("✓ Randomization complete\n")
    
    # Step 3: Initialize statistics tracking
    stats = {
        "start_time": datetime.now(),
        "total_runs": len(design_matrix),
        "successful_runs": 0,
        "failed_runs": 0,
        "by_temperature": {label: {"successful": 0, "failed": 0} for label in temperature_dict.keys()},
        "by_block": {"S": {"successful": 0, "failed": 0}, 
                     "M": {"successful": 0, "failed": 0}, 
                     "F": {"successful": 0, "failed": 0}}
    }
    
    # Step 4: Execute experiment
    with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = [
            'run_id', 'timestamp',
            'block', 'question_number', 'question_type', 'question_text',
            'temperature_level', 'temperature_value',
            'answer',
            'status', 'error_message', 'iteration_count', 'retry_count',
            'tool_calls_count', 'search_queries',
            'C1_correctness', 'C2_coverage', 'C3_citation_presence',
            'C4_citation_valid', 'C5_clarity', 'RGS',
            'scored_by', 'scoring_notes'
        ]
        
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        print("="*70)
        print("RCBD TEMPERATURE EXPERIMENT")
        print("="*70)
        print(f"Design: Randomized Complete Block Design")
        print(f"Treatment: Temperature ({len(temperature_dict)} levels)")
        print(f"Blocking: Question Type (3 blocks: S, M, F)")
        print(f"Total runs: {len(design_matrix)}")
        print(f"RGS Formula: C1 × (C2+C3+C4+C5) / 4")
        print("="*70 + "\n")
        
        for run_id, run_spec in enumerate(design_matrix, 1):
            question_number = run_spec["question_number"]
            question_text = run_spec["question_text"]
            question_type = run_spec["question_type"]
            block = run_spec["block"]
            temp_label = run_spec["temperature_level"]
            temp_value = run_spec["temperature_value"]
            
            print(f"Run {run_id:2d}/{len(design_matrix)} | Block={block} | Q{question_number:2d} | T={temp_value} ({temp_label})")
            print(f"Q: {question_text[:60]}...")
            
            timestamp = datetime.now().isoformat()
            
            # Execute single run
            result = answer_clue_question(
                question_text,
                temperature_value=temp_value,
                max_iterations=5,
                max_retries=3,
                retry_delay=5
            )
            
            # Track results
            if result["error"] is None:
                status = "SUCCESS"
                stats["successful_runs"] += 1
                stats["by_temperature"][temp_label]["successful"] += 1
                stats["by_block"][block]["successful"] += 1
                print(f"✅ Success")
                print(f"A: {result['answer'][:60]}...")
            else:
                status = "ERROR"
                stats["failed_runs"] += 1
                stats["by_temperature"][temp_label]["failed"] += 1
                stats["by_block"][block]["failed"] += 1
                print(f"❌ Error: {result['error'][:60]}...")
            
            # Write to CSV
            writer.writerow({
                'run_id': run_id,
                'timestamp': timestamp,
                'block': block,
                'question_number': question_number,
                'question_type': question_type,
                'question_text': question_text,
                'temperature_level': temp_label,
                'temperature_value': temp_value,
                'answer': result["answer"],
                'status': status,
                'error_message': result["error"] or '',
                'iteration_count': result["iterations"],
                'retry_count': result["retries"],
                'tool_calls_count': len(result.get("tool_calls", [])),
                'search_queries': "; ".join([tc["query"] for tc in result.get("tool_calls", [])]),
                'C1_correctness': '',
                'C2_coverage': '',
                'C3_citation_presence': '',
                'C4_citation_valid': '',
                'C5_clarity': '',
                'RGS': '',
                'scored_by': '',
                'scoring_notes': ''
            })
            
            # Delay between runs (except last)
            if run_id < len(design_matrix):
                print(f"⏳ {delay_seconds}s...\n")
                time.sleep(delay_seconds)
            else:
                print()
    
    # Calculate final statistics
    stats["end_time"] = datetime.now()
    stats["duration_minutes"] = (stats["end_time"] - stats["start_time"]).total_seconds() / 60
    
    # Print summary
    print("\n" + "="*70)
    print("EXPERIMENT COMPLETE")
    print("="*70)
    print(f"Total runs: {stats['total_runs']}")
    print(f"Successful: {stats['successful_runs']}")
    print(f"Failed: {stats['failed_runs']}")
    print(f"Duration: {stats['duration_minutes']:.1f} min")
    
    print(f"\nBy Temperature:")
    for temp_label in temperature_dict.keys():
        temp_stats = stats["by_temperature"][temp_label]
        print(f"  {temp_label:8s}: {temp_stats['successful']:2d} ✓, {temp_stats['failed']:2d} ✗")
    
    print(f"\nBy Block (Question Type):")
    for block in ["S", "M", "F"]:
        block_stats = stats["by_block"][block]
        print(f"  Block {block}: {block_stats['successful']:2d} ✓, {block_stats['failed']:2d} ✗")
    
    print("="*70)
    print(f"\n📁 {csv_filename}")
    print(f"📊 Next steps:")
    print(f"   1. Score C1-C5 columns")
    print(f"   2. Calculate RGS")
    print(f"   3. Import to JMP")
    print(f"   4. Run ANOVA with blocking on question_type\n")
    
    return stats

print("✓ RCBD experiment function defined")

✓ RCBD experiment function defined


## 8. Post-Experiment Helper Functions

In [38]:
def create_scoring_template(input_csv, output_csv="scoring_template_RCBD.csv"):
    """
    Create simplified CSV for manual scoring.
    """
    df = pd.read_csv(input_csv)
    df_success = df[df['status'] == 'SUCCESS'].copy()
    
    scoring_df = df_success[[
        'run_id', 'block', 'question_number', 'question_type',
        'question_text', 'temperature_level', 'temperature_value', 'answer'
    ]].copy()
    
    scoring_df['C1_correctness'] = ''
    scoring_df['C2_coverage'] = ''
    scoring_df['C3_citation_presence'] = ''
    scoring_df['C4_citation_valid'] = ''
    scoring_df['C5_clarity'] = ''
    scoring_df['scoring_notes'] = ''
    
    scoring_df.to_csv(output_csv, index=False)
    print(f"✅ Scoring template: {output_csv}")
    print(f"   {len(scoring_df)} responses to score")
    print(f"   Grouped by block for easier scoring")

def merge_scores(experiment_csv, scored_csv, output_csv="final_RCBD_with_rgs.csv"):
    """
    Merge scored C1-C5 back and calculate RGS.
    RGS = C1 × (C2 + C3 + C4 + C5) / 4
    """
    df_exp = pd.read_csv(experiment_csv)
    df_scored = pd.read_csv(scored_csv)
    
    df_scored['RGS'] = (
        df_scored['C1_correctness'] * 
        (df_scored['C2_coverage'] + df_scored['C3_citation_presence'] + 
         df_scored['C4_citation_valid'] + df_scored['C5_clarity']) / 4
    )
    
    df_merged = df_exp.merge(
        df_scored[['run_id', 'C1_correctness', 'C2_coverage',
                   'C3_citation_presence', 'C4_citation_valid',
                   'C5_clarity', 'RGS', 'scoring_notes']],
        on='run_id', how='left', suffixes=('', '_scored')
    )
    
    for col in ['C1_correctness', 'C2_coverage', 'C3_citation_presence',
                'C4_citation_valid', 'C5_clarity', 'RGS', 'scoring_notes']:
        if f'{col}_scored' in df_merged.columns:
            df_merged[col] = df_merged[f'{col}_scored'].combine_first(df_merged[col])
            df_merged.drop(f'{col}_scored', axis=1, inplace=True)
    
    df_merged.to_csv(output_csv, index=False)
    print(f"✅ Final CSV: {output_csv}")
    print(f"\nRGS Statistics by Temperature:")
    print(df_merged[df_merged['status']=='SUCCESS'].groupby('temperature_level')['RGS'].describe())
    print(f"\nRGS Statistics by Block:")
    print(df_merged[df_merged['status']=='SUCCESS'].groupby('block')['RGS'].describe())

print("✓ Post-experiment helpers defined")

✓ Post-experiment helpers defined


## 9. RUN RCBD EXPERIMENT

**Before running:**
- Ensure all previous cells have run successfully
- Check that `questions` variable contains 30 questions (10 per type: S, M, F)
- Verify API keys are working

**Expected runtime:** ~7-10 minutes (90 runs × 3s delay + processing time)

**RCBD guarantees:**
- Each question appears exactly 3 times (once per temperature)
- Run order is completely randomized
- No systematic confounding of temperature with execution order

In [39]:
# Run the RCBD experiment
stats = run_RCBD_temperature_experiment(
    questions_data=questions,
    temperature_dict=temp,
    csv_filename="clue_temperature_RCBD.csv",
    delay_seconds=3
)


CREATING RCBD DESIGN MATRIX
Design matrix created: 93 runs
  31 questions × 3 temperatures

Blocking structure:
  Block F: 11 questions × 3 temps = 33 runs
  Block M: 10 questions × 3 temps = 30 runs
  Block S: 10 questions × 3 temps = 30 runs

🔀 Randomizing all 93 runs...
✓ Randomization complete

RCBD TEMPERATURE EXPERIMENT
Design: Randomized Complete Block Design
Treatment: Temperature (3 levels)
Blocking: Question Type (3 blocks: S, M, F)
Total runs: 93
RGS Formula: C1 × (C2+C3+C4+C5) / 4

Run  1/93 | Block=F | Q 4 | T=0.8 (high)
Q: How can I stop grandma ju from broadcasting her cards?...
--- Iteration 1 ---
Stop reason: end_turn
✅ Success
A: I only answer questions about the game Clue. 

If you're ask...
⏳ 3s...

Run  2/93 | Block=F | Q10 | T=0.3 (low)
Q: Logically if a player is also Mr Body, how can s/he play the...
--- Iteration 1 ---
Stop reason: tool_use
Claude wants to use: search_clue_instructions
With query: Mr. Body player count setup
Tool result preview: [
  {
    "chu

## 10. Post-Experiment Workflow

After experiment completes:

1. **Create scoring template**
2. **Score responses** (manually or with LLM-as-judge)
3. **Merge scores** and calculate RGS
4. **Import to JMP** for blocked ANOVA

**JMP Analysis Setup:**
- Response: RGS
- Fixed Effect: temperature_value (or temperature_level)
- Blocking Variable: block (or question_type)
- Model: Y = μ + Temperature + Block + ε

In [40]:
# Step 1: Create scoring template
create_scoring_template(
    input_csv="clue_temperature_RCBD.csv",
    output_csv="responses_to_score_RCBD.csv"
)

✅ Scoring template: responses_to_score_RCBD.csv
   93 responses to score
   Grouped by block for easier scoring


In [41]:
# Step 2: After scoring in Excel, merge back and calculate RGS
merge_scores(
    experiment_csv="clue_temperature_RCBD.csv",
    scored_csv="responses_to_score_RCBD.csv",
    output_csv="final_RCBD_with_rgs.csv"
)

print("\n📊 Ready for JMP Analysis!")
print("Import final_RCBD_with_rgs.csv into JMP")
print("\nJMP Model Setup:")
print("  Y (Response): RGS")
print("  X (Treatment): temperature_value")
print("  Block: block (or question_type)")
print("  Model Effects: temperature_value + block")

✅ Final CSV: final_RCBD_with_rgs.csv

RGS Statistics by Temperature:
                   count  mean  std  min  25%  50%  75%  max
temperature_level                                           
high                 0.0   NaN  NaN  NaN  NaN  NaN  NaN  NaN
low                  0.0   NaN  NaN  NaN  NaN  NaN  NaN  NaN
medium               0.0   NaN  NaN  NaN  NaN  NaN  NaN  NaN

RGS Statistics by Block:
       count  mean  std  min  25%  50%  75%  max
block                                           
F        0.0   NaN  NaN  NaN  NaN  NaN  NaN  NaN
M        0.0   NaN  NaN  NaN  NaN  NaN  NaN  NaN
S        0.0   NaN  NaN  NaN  NaN  NaN  NaN  NaN

📊 Ready for JMP Analysis!
Import final_RCBD_with_rgs.csv into JMP

JMP Model Setup:
  Y (Response): RGS
  X (Treatment): temperature_value
  Block: block (or question_type)
  Model Effects: temperature_value + block


In [ ]:
# Print what's ACTUALLY in the retriever
print("What retriever sees as CHUNK_1:")
print(contextualized_chunks[0])
print("\n" + "="*70 + "\n")

print("Original CHUNK_1:")
print(chunks[0])



What retriever sees as CHUNK_1:
This is the title page and header information of the CLUE Classic Detective Game Rules, establishing the game's basic parameters including player count, age range, and copyright information before the objective is detailed.
# CLUE Classic Detective Game Rules

For 3 to 6 players / Ages 8 to adult  
Rules ©1986 Hasbro, Inc. Printed in U.S.A.



Original CHUNK_1:
# CLUE Classic Detective Game Rules

For 3 to 6 players / Ages 8 to adult  
Rules ©1986 Hasbro, Inc. Printed in U.S.A.

Original chunks: 8
Contextualized chunks: 8

Original chunk 1 length: 119
Contextualized chunk 1 length: 343
